# 16 — HuggingFace Upload & Model Card

This notebook is the final step in the Medical SLM pipeline.
We push all artifacts to HuggingFace Hub so the model is:

- **Reproducible** — anyone can load your exact tokenizer and weights
- **Shareable** — colleagues and the community can evaluate and build on your model
- **Documented** — the model card explains training, limitations, and appropriate use

**What we upload:**
| Artifact | Source | Destination |
|---|---|---|
| BPE tokenizer files | `cfg.MED_TOKENIZER_DIR` | `cfg.MED_HF_TOKENIZER_REPO` |
| SFT model weights | `cfg.MED_SFT_FINAL_CKPT` | `cfg.MED_HF_MODEL_REPO` |
| DPO model weights (optional) | `cfg.MED_DPO_FINAL_CKPT` | `cfg.MED_HF_MODEL_REPO` |
| Model card (README.md) | Generated here | `cfg.MED_HF_MODEL_REPO` |

## 1 — Setup

In [ ]:
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
# !pip install huggingface_hub
from huggingface_hub import HfApi, login
import huggingface_hub
print(f"huggingface_hub version: {huggingface_hub.__version__}")

## 2 — What to Upload

### Tokenizer artifacts
- `vocab.json` — the BPE vocabulary (token → ID mapping)
- `merges.txt` — the BPE merge rules (in order of frequency during training)

These two files are the complete specification of the medical tokenizer.
Anyone with these files + the `tokenizers` library can reproduce exact tokenization.

### Model weights
- `sft_final.pt` — the SFT model state dict (primary artifact)
- `dpo_final.pt` — the DPO model (optional; best MCQ accuracy)

We upload as raw `.pt` files rather than converting to HuggingFace format because our model
is a custom architecture (not a standard `transformers` model class). A future step would
be to implement a `PreTrainedModel` wrapper for full HuggingFace compatibility.

### Model card (README.md)
The model card documents training procedure, datasets, evaluation results, and usage.
HuggingFace renders it as the repository landing page.
It must include a medical disclaimer since this is a research model.

## 3 — Set HuggingFace Repo Names

Before uploading, set `cfg.MED_HF_MODEL_REPO` and `cfg.MED_HF_TOKENIZER_REPO` in `config.py`.

**Format:** `"your-hf-username/repo-name"`

Example:
```python
MED_HF_MODEL_REPO     = "jsmith/medical-slm-25m"
MED_HF_TOKENIZER_REPO = "jsmith/medical-slm-tokenizer"
```

You can also override here for this session only (does not persist to config.py):
```python
import config as cfg
cfg.MED_HF_MODEL_REPO     = "your-username/medical-slm-25m"
cfg.MED_HF_TOKENIZER_REPO = "your-username/medical-slm-tokenizer"
```

In [ ]:
import config as cfg

# ── EDIT THESE VALUES ────────────────────────────────────────────────────
# Uncomment and fill in your HuggingFace username before running this cell.
# cfg.MED_HF_MODEL_REPO     = "your-username/medical-slm-25m"
# cfg.MED_HF_TOKENIZER_REPO = "your-username/medical-slm-tokenizer"
# ─────────────────────────────────────────────────────────────────────────

print(f"Model repo     : {cfg.MED_HF_MODEL_REPO}")
print(f"Tokenizer repo : {cfg.MED_HF_TOKENIZER_REPO}")

assert cfg.MED_HF_MODEL_REPO and "/" in cfg.MED_HF_MODEL_REPO, \
    "Set cfg.MED_HF_MODEL_REPO to 'username/repo-name' before proceeding."
assert cfg.MED_HF_TOKENIZER_REPO and "/" in cfg.MED_HF_TOKENIZER_REPO, \
    "Set cfg.MED_HF_TOKENIZER_REPO to 'username/repo-name' before proceeding."

In [ ]:
# Authenticate with HuggingFace Hub.
# On Colab: you can set HF_TOKEN as a Colab secret and read it from the environment.
# Locally: run `huggingface-cli login` once in your terminal.
import os

HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in via HF_TOKEN environment variable.")
else:
    print("HF_TOKEN not found in environment.")
    print("Run `login()` interactively or set the HF_TOKEN secret in Colab.")
    # login()   # uncomment to login interactively

api = HfApi()
print("HfApi ready.")

## 4 — Upload Tokenizer

`HfApi.upload_folder()` uploads all files in a local directory to a Hub repository.
We create the repo first if it does not exist, then upload the tokenizer directory.

We upload to `repo_type="model"` (not `"dataset"`) because the tokenizer is logically part
of the model — it defines the vocabulary the weights were trained on.

In [ ]:
import os
import config as cfg

# Verify tokenizer files exist before uploading
for path in [cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  Found: {path}  ({size_kb:.1f} KB)")
    else:
        print(f"  MISSING: {path}")
        print("  → Run notebook 11 to train and save the medical tokenizer.")
        raise FileNotFoundError(path)

In [ ]:
# Create the tokenizer repository if it does not exist
try:
    api.create_repo(repo_id=cfg.MED_HF_TOKENIZER_REPO, repo_type="model", exist_ok=True)
    print(f"Repository ready: https://huggingface.co/{cfg.MED_HF_TOKENIZER_REPO}")
except Exception as e:
    print(f"Error creating repo: {e}")
    raise

In [ ]:
# Upload all files in the tokenizer directory
api.upload_folder(
    folder_path=cfg.MED_TOKENIZER_DIR,
    repo_id=cfg.MED_HF_TOKENIZER_REPO,
    repo_type="model",
    commit_message="Upload medical BPE tokenizer (trained on medical_meadow_textbooks)",
)
print(f"\nTokenizer uploaded to: https://huggingface.co/{cfg.MED_HF_TOKENIZER_REPO}")

## 5 — Upload Model Weights

We upload to a single model repository. The SFT checkpoint is the primary artifact;
the DPO checkpoint is optional (upload if it improved MCQ accuracy in notebook 15).

Each `.pt` file is uploaded as a single file (not a folder) since they are individual artifacts.

In [ ]:
# Create the model repository
try:
    api.create_repo(repo_id=cfg.MED_HF_MODEL_REPO, repo_type="model", exist_ok=True)
    print(f"Repository ready: https://huggingface.co/{cfg.MED_HF_MODEL_REPO}")
except Exception as e:
    print(f"Error creating repo: {e}")
    raise

In [ ]:
import os
import config as cfg

# Upload SFT weights
if os.path.exists(cfg.MED_SFT_FINAL_CKPT):
    size_mb = os.path.getsize(cfg.MED_SFT_FINAL_CKPT) / 1e6
    print(f"Uploading SFT checkpoint ({size_mb:.1f} MB)...")
    api.upload_file(
        path_or_fileobj=cfg.MED_SFT_FINAL_CKPT,
        path_in_repo="sft_final.pt",
        repo_id=cfg.MED_HF_MODEL_REPO,
        repo_type="model",
        commit_message="Upload SFT model weights (medical_meadow_medqa + pubmed_qa)",
    )
    print(f"  → Uploaded sft_final.pt")
else:
    print(f"SFT checkpoint not found: {cfg.MED_SFT_FINAL_CKPT}")
    print("  → Run notebook 13 to complete SFT training.")

In [ ]:
# Upload DPO weights (optional — only if notebook 15 produced an improvement)
if os.path.exists(cfg.MED_DPO_FINAL_CKPT):
    size_mb = os.path.getsize(cfg.MED_DPO_FINAL_CKPT) / 1e6
    print(f"Uploading DPO checkpoint ({size_mb:.1f} MB)...")
    api.upload_file(
        path_or_fileobj=cfg.MED_DPO_FINAL_CKPT,
        path_in_repo="dpo_final.pt",
        repo_id=cfg.MED_HF_MODEL_REPO,
        repo_type="model",
        commit_message="Upload DPO model weights (preference-optimised on MedQA choices)",
    )
    print(f"  → Uploaded dpo_final.pt")
else:
    print(f"DPO checkpoint not found: {cfg.MED_DPO_FINAL_CKPT}")
    print("  → Skipping DPO upload. Run notebook 15 to generate DPO weights.")

## 6 — Generate Model Card

A good model card is as important as the model itself. It tells users:
- What the model can and cannot do
- How it was trained (data, stages, hyperparameters)
- How to load and use it
- What the evaluation results were
- What the risks and limitations are

For medical models, the **disclaimer** is mandatory: this model is for research only and
must not be used for clinical decisions.

**Instructions:**  
Fill in the placeholder values (marked with `TODO`) with your actual results from notebook 14
before uploading.

In [ ]:
import os
import config as cfg

# ── Fill in your actual MCQ accuracy numbers from notebook 14 here ────────
# Replace TODO values with your results before uploading the model card.
TS_MCQ_ACC = "TODO (e.g., 24.5%)"   # TinyStories base accuracy from nb14
DAPT_MCQ_ACC  = "TODO (e.g., 31.2%)"   # DAPT accuracy from nb14
SFT_MCQ_ACC   = "TODO (e.g., 48.7%)"   # SFT accuracy from nb14
DPO_MCQ_ACC   = "TODO (e.g., 51.3%)"   # DPO accuracy from nb15
# ─────────────────────────────────────────────────────────────────────────

MODEL_CARD = f"""---
language: en
license: apache-2.0
tags:
  - medical
  - language-model
  - gpt
  - bpe
  - usmle
  - pubmedqa
datasets:
  - medalpaca/medical_meadow_textbooks
  - medalpaca/medical_meadow_medqa
  - pubmed_qa
---

# Medical SLM — 25M Parameter GPT-style Language Model

A small language model (SLM) trained end-to-end for medical question answering.
Built from scratch in PyTorch as a learning project — every component is hand-written
and documented for pedagogical clarity.

## Model Description

| Property | Value |
|---|---|
| Architecture | GPT (decoder-only transformer) |
| Parameters | ~25M |
| Layers | {cfg.N_LAYERS} |
| Attention heads | {cfg.N_HEADS} |
| Embedding dim | {cfg.D_MODEL} |
| Vocabulary size | {cfg.VOCAB_SIZE:,} |
| Context length | 512 tokens |
| Tokenizer | Custom BPE (trained on medical textbooks) |

## Training Stages

This model was trained through three successive stages:

### Stage 1: Domain-Adaptive Pretraining (DAPT)
- **Starting point:** Cosmopedia-v2 pretrained checkpoint (~25M parameters)
- **Dataset:** `medalpaca/medical_meadow_textbooks` — {cfg.MED_TEXTBOOK_DATA_SIZE:,} training documents
- **Steps:** {cfg.MED_DAPT_MAX_STEPS:,}
- **Context length:** 512 tokens
- **Key decision:** Transformer block weights transferred from TinyStories; token embeddings
  and LM head reinitialized for the new medical vocabulary

### Stage 2: Supervised Fine-Tuning (SFT)
- **Dataset:** Mix of `medalpaca/medical_meadow_medqa` ({cfg.MED_MEDQA_TRAIN_SIZE:,} examples)
  and `pubmed_qa` labeled split ({cfg.MED_PUBMEDQA_TRAIN_SIZE:,} examples), blended 80/20
- **Steps:** {cfg.MED_SFT_MAX_STEPS:,}
- **Technique:** Response masking — loss computed only on answer tokens, not question tokens

### Stage 3: Direct Preference Optimization (DPO)
- **Preference data:** MedQA correct answers as chosen, wrong options as rejected
- **Pairs:** Up to 8,000 (prompt, chosen, rejected) triples
- **Steps:** {cfg.MED_DPO_MAX_STEPS:,}
- **Beta:** {cfg.MED_DPO_BETA}

## Evaluation Results

USMLE-style MCQ accuracy on held-out MedQA test examples (likelihood scoring):

| Model | MCQ Accuracy |
|---|---|
| TinyStories base (no medical training) | {TS_MCQ_ACC} |
| After DAPT | {DAPT_MCQ_ACC} |
| After SFT | {SFT_MCQ_ACC} |
| After DPO | {DPO_MCQ_ACC} |

**Note:** These results are from a 25M parameter model trained on a single T4 GPU.
They are not comparable to larger medical LLMs (BioMedLM, MedPaLM, etc.).

## How to Load

```python
import sys
sys.path.insert(0, "/path/to/slm-learning")

import torch
from model.gpt import GPT, GPTConfig
from tokenizer.preprocess import load_tokenizer
from generation.sampler import generate, encode_prompt

# Load tokenizer
tokenizer = load_tokenizer("vocab.json", "merges.txt")

# Load model
config = GPTConfig(vocab_size=8192, block_size=512, n_layer=8, n_head=8, n_embd=512)
model  = GPT(config)
model.load_state_dict(torch.load("sft_final.pt", map_location="cpu"))
model.eval()

# Generate
prompt = "The mechanism of action of beta-blockers involves"
x      = encode_prompt(prompt, tokenizer, device="cpu")
output = generate(model, x, tokenizer, block_size=512, max_new_tokens=150)
print(tokenizer.decode(output[0].tolist()))
```

## Datasets Used

- [`medalpaca/medical_meadow_textbooks`](https://huggingface.co/datasets/medalpaca/medical_meadow_textbooks) — open-access medical textbook passages
- [`medalpaca/medical_meadow_medqa`](https://huggingface.co/datasets/medalpaca/medical_meadow_medqa) — USMLE-style multiple choice questions
- [`pubmed_qa`](https://huggingface.co/datasets/pubmed_qa) — biomedical research question answering

## Limitations

- **Small model:** At 25M parameters, this model cannot match the depth of reasoning
  of larger medical LLMs. It frequently makes factual errors.
- **Limited training data:** {cfg.MED_TEXTBOOK_DATA_SIZE:,} textbook examples is a fraction
  of the biomedical literature. The model lacks knowledge of recent developments.
- **No factual grounding:** The model generates text that sounds plausible but may be wrong.
  All outputs must be verified against authoritative medical sources.
- **English only:** Training data is entirely in English.
- **USMLE bias:** SFT training on USMLE-style questions may bias the model toward
  US clinical practice guidelines.

## Medical Disclaimer

> **This model is for educational and research purposes only.**
> It is NOT intended for clinical use or medical decision-making.
> Do not use this model to diagnose, treat, or manage any medical condition.
> Always consult a qualified healthcare professional for medical advice.
> The authors accept no liability for any harm arising from the use of this model.

## Citation

If you use this model in research, please cite:
```
@misc{{medical-slm-25m,
  title  = {{Medical SLM — A 25M Parameter Medical Language Model}},
  year   = {{2026}},
  note   = {{Trained end-to-end as part of the slm-from-scratch learning project}}
}}
```
"""

# Save the model card locally
readme_path = os.path.join(cfg.MED_CHECKPOINT_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(MODEL_CARD)

print(f"Model card written to: {readme_path}")
print(f"Length: {len(MODEL_CARD):,} characters")
print()
print("--- Preview (first 500 chars) ---")
print(MODEL_CARD[:500])

In [ ]:
# Upload the model card as README.md to the model repository
api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=cfg.MED_HF_MODEL_REPO,
    repo_type="model",
    commit_message="Add model card with training details and medical disclaimer",
)
print(f"Model card uploaded to: https://huggingface.co/{cfg.MED_HF_MODEL_REPO}")

## 7 — Verify Upload

List all files in both repositories to confirm everything was uploaded correctly.

In [ ]:
import config as cfg

print("=" * 60)
print(f"Tokenizer repo: {cfg.MED_HF_TOKENIZER_REPO}")
print(f"URL: https://huggingface.co/{cfg.MED_HF_TOKENIZER_REPO}")
print()
try:
    tok_files = api.list_repo_files(repo_id=cfg.MED_HF_TOKENIZER_REPO, repo_type="model")
    for f in tok_files:
        print(f"  {f}")
except Exception as e:
    print(f"  Could not list files: {e}")

print()
print(f"Model repo: {cfg.MED_HF_MODEL_REPO}")
print(f"URL: https://huggingface.co/{cfg.MED_HF_MODEL_REPO}")
print()
try:
    model_files = api.list_repo_files(repo_id=cfg.MED_HF_MODEL_REPO, repo_type="model")
    for f in model_files:
        print(f"  {f}")
except Exception as e:
    print(f"  Could not list files: {e}")

In [ ]:
import config as cfg

print("=" * 60)
print("Upload complete. Share these URLs:")
print()
print(f"  Tokenizer : https://huggingface.co/{cfg.MED_HF_TOKENIZER_REPO}")
print(f"  Model     : https://huggingface.co/{cfg.MED_HF_MODEL_REPO}")
print("=" * 60)
print()
print("Remember to fill in the TODO accuracy values in the model card!")
print("  → Re-run section 6 with actual numbers from notebooks 14 and 15,")
print("    then re-upload the README.md.")

## Pipeline Complete

You have now completed the full Medical SLM pipeline:

| Notebook | Stage | Key output |
|---|---|---|
| 10 | Data exploration | Understanding of 3 datasets + tokenizer motivation |
| 11 | Tokenizer + corpus | Medical BPE tokenizer + tokenized train/val/test splits |
| 12 | DAPT | `cfg.MED_DAPT_FINAL_CKPT` — domain-adapted model |
| 13 | SFT | `cfg.MED_SFT_FINAL_CKPT` — instruction-following model |
| 14 | Evaluation | MCQ accuracy + perplexity benchmark across all checkpoints |
| 15 | DPO | `cfg.MED_DPO_FINAL_CKPT` — preference-optimised model |
| 16 | HF Upload | Public model + tokenizer + model card on HuggingFace Hub |

**What to explore next:**
- Apply LoRA (PEFT) in a notebook 17 to fine-tune with even less memory
- Quantize the model to INT8 with bitsandbytes for faster inference
- Build a simple Gradio demo that serves the SFT model for medical Q&A